# Customer Satisfaction Dataset

In [ ]:
import pandas as pd
from sklearn.feature_selection import mutual_info_regression
import seaborn as sns
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

**Step-1:** Import the dataset

In [ ]:
df = pd.read_csv("/kaggle/input/cs1234/Santander Customer Satisfaction_train.csv")
print(df.shape)
target_column = 0
# Check if all features are numeric
features_df = df.drop(columns=target_column) if target_column else df
all_numeric = all(pd.api.types.is_numeric_dtype(features_df[col]) for col in features_df.columns)
print("All features numeric:", all_numeric)
     
df.head()

**Step-2:** Handling the Null values in the dataset

In [ ]:
df.isnull().sum()

In [ ]:
#Drop the null values if you have more than 30% of Null
df.dropna(axis=1, thresh=0.7*len(df), inplace=True)

#Fill the null values
df = df.fillna(df.median())

In [ ]:
df.head()

**Step-3:** Splitting the Dataset

In [ ]:
X = df.drop('TARGET', axis=1)
y = df['TARGET']

**Step-4:** Applying Constant Feature

In [ ]:
constant_feature = [col for col in X.columns if X[col].nunique == 1]
print(constant_feature)

Step-5: Quasi Constant Feature

In [ ]:
quasi_constant_feature = []
threshold = 0.98

for col in X.columns:
    freq = X[col].value_counts(normalize=True).iloc[0]
    if freq >= threshold:
        quasi_constant_feature.append(col)

print("quasi_constant_feature: ", quasi_constant_feature)
print("\nTotal: ", len(quasi_constant_feature))

X = X.drop(columns=quasi_constant_feature, axis=1)

**Step-6:** Duplicate Features

In [ ]:
train_T = X.T
duplicate_features = train_T[train_T.duplicated()].index.tolist()
print(duplicate_features)
print("\nTotal Duplicate Features: ", len(duplicate_features))
X = X.drop(columns=duplicate_features, axis=1)

**Step-7:** Correlation

In [ ]:
corr_matrix = X.corr()
corr_threshold = 0.98
corr_pairs = set()

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i,j] > corr_threshold):
            corr_pairs.add(corr_matrix.columns[i])

print("Correlation Features: ",corr_pairs)
print("\nTotal: ", len(corr_pairs))

X = X.drop(columns=corr_pairs, axis=1)


In [ ]:
#Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap-")
plt.show()

**Step-8:** Mututal Information Regression

In [ ]:
mi_scores = mutual_info_regression(X, y, discrete_features=False)
mi_df = pd.Series(mi_scores, index=X.columns)
mi_df = mi_df.sort_values(ascending=False)


low_importance = mi_df[mi_df < 0.01].index.tolist()

print("Mututal Information: ",low_importance)
print("\nTotal: ", len(low_importance))

X = X.drop(columns=low_importance,axis=1)

In [ ]:
#Chi Square cannot be applied since the output feature is numerical

In [ ]:
from sklearn.feature_selection import f_regression
import pandas as pd

f_scores, p_values = f_regression(X, y)


anova_df = pd.DataFrame({
    'F_Score': f_scores,
    'P_Value': p_values
}, index=X.columns)

anova_df = anova_df.sort_values(by='F_Score', ascending=False)

insignificant_features = anova_df[anova_df['P_Value'] > 0.05].index.tolist()

print("Dropping features with weak linear relationship:", insignificant_features)
print("Total: ", len(insignificant_features))

X = X.drop(columns=insignificant_features)
